# Dashcam Safety Copilot: train and evaluate the detector on Kaggle

This notebook only *runs* the repo's scripts; all settings live in `ml/configs/`.

**Before you run it**
1. Notebook settings (right panel): **Accelerator = GPU T4 x2** (or P100), **Internet = on**.
2. **Add Input** → your private Kaggle dataset containing the merged folder
   (`data.yaml`, `images/`, `labels/`) that you built locally with `dashcam_ml.merge`.
3. Optional, to resume an interrupted run: also add the *output* of the previous run of this
   notebook as an input (see the resume cell).

Kaggle gives about 30 GPU hours per week and a session stops after 12 hours.
Run with **Save Version → Save & Run All** so training continues after you close the tab.

In [ ]:
!nvidia-smi
!git clone -q https://github.com/iamshubhamk/dashcam-safety-copilot.git
%cd dashcam-safety-copilot
# Kaggle already has torch with CUDA; install only ultralytics (pinned) and our package without
# touching Kaggle's numpy/pillow.
!pip install -q ultralytics==8.4.197
!pip install -q --no-deps -e ml

In [ ]:
import glob

cands = [p for p in glob.glob('/kaggle/input/**/data.yaml', recursive=True)]
print(cands)
assert len(cands) == 1, 'attach exactly one merged dataset (a folder with data.yaml, images/, labels/)'
DATA = cands[0]

## Train
`yolo26s` fine-tuned from COCO weights. Watch the first epoch's time and multiply by `epochs`
to see whether the run fits the 12-hour session; if not, lower `train.epochs` (e.g. `--set train.epochs=40`)
or plan to resume in a second session. Use `train.device=0,1` on the T4 x2 accelerator.

In [ ]:
!python -m dashcam_ml.train --config ml/configs/train/yolo26s_idd.yaml \
    --set data={DATA} train.project=/kaggle/working/runs train.device=0,1 \
    --export-best /kaggle/working/weights/best.pt

### Resume (only if the previous session was cut off)
Attach the previous run's output, then point `--resume` at its `last.pt`. Ultralytics restores the
optimizer and epoch counter, so it continues where it stopped.

In [ ]:
# LAST = glob.glob('/kaggle/input/**/runs/yolo26s_idd*/weights/last.pt', recursive=True)[0]
# !mkdir -p /kaggle/working/runs && cp -r $(dirname $(dirname {LAST})) /kaggle/working/runs/
# !python -m dashcam_ml.train --resume /kaggle/working/runs/yolo26s_idd/weights/last.pt

## Evaluate: fine-tuned vs stock COCO baseline
Both models are scored by the same code on the same val images. Writes
`/kaggle/working/results/phase1_eval.{json,md}`; download them and commit them to `results/` in the repo.

In [ ]:
!python -m dashcam_ml.evaluate --weights /kaggle/working/weights/best.pt --baseline yolo26s.pt \
    --data {DATA} --device 0 --crosscheck --out /kaggle/working/results/phase1_eval

## Keep
From the notebook's **Output** tab download `weights/best.pt` (keep it out of git; the IDD licence is
non-commercial) and `results/phase1_eval.json` + `.md` (commit these).